In [1]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [2]:
model_path = "../simple_conv.onnx"

In [3]:
# A small CNN exercising every layer type currently supported by the Rust runtime:
# Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear -> Sigmoid -> Linear -> Tanh -> Linear -> Softmax
#
# Input is (1, 1, 4, 4): batch size 1, 1 channel, 4x4 spatial. Both convs use
# kernel_size=3, stride=1, padding=1 so spatial dims stay 4x4 throughout ("same" padding),
# which keeps the flattened size (8 * 4 * 4 = 128) easy to check by hand.


class SimpleConvModel(nn.Module):
    def __init__(self):
        super(SimpleConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act_2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act_3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act_4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, 5)
        self.act_5_softmax = nn.Softmax(dim=1)  # Apply softmax along the feature dimension

    def forward(self, x):
        output = self.conv1(x)
        output = self.act_1_relu(output)
        output = self.conv2(output)
        output = self.act_2_relu(output)
        output = self.flatten(output)
        output = self.linear1(output)
        output = self.act_3_sigmoid(output)
        output = self.linear2(output)
        output = self.act_4_tanh(output)
        output = self.linear3(output)
        output = self.act_5_softmax(output)
        return output


# Example usage
model = SimpleConvModel()
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleConvModel(
  (conv1): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_1_relu): ReLU()
  (conv2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_2_relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=128, out_features=20, bias=True)
  (act_3_sigmoid): Sigmoid()
  (linear2): Linear(in_features=20, out_features=15, bias=True)
  (act_4_tanh): Tanh()
  (linear3): Linear(in_features=15, out_features=5, bias=True)
  (act_5_softmax): Softmax(dim=1)
)
Model weights:
conv1.weight: torch.Size([4, 1, 3, 3])
  Weight values (first 5): tensor([ 0.1425, -0.0177, -0.1640, -0.1665, -0.2294], grad_fn=<SliceBackward0>)
conv1.bias: torch.Size([4])
  Bias values (first 5): tensor([-0.0210, -0.3103,  0.2899,  0.2043], grad_fn=<SliceBackward0>)
conv2.weight: torch.Size([8, 4, 3, 3])
  Weight values (first 5): tensor([ 0.0682, -0.0030,  0.1487,  0.0910,  0.0344], grad_fn=<SliceBackward0>)
conv2.bias: torch.Size([8])
  

In [4]:
# export to onnx
dummy_input = torch.randn(1, 1, 4, 4)
onnx.export(model, dummy_input, model_path, export_params=True, opset_version=11)

In [5]:
# run the model with pytorch
# 4x4 single-channel "image" with values 1..16, so it's easy to eyeball against the Rust output
input_data = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
with torch.no_grad():
    output = model(input_data)
print(input_data)
print(output)

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])
tensor([[0.1888, 0.2796, 0.2389, 0.1913, 0.1014]])


In [7]:
class HugeLinearModel(nn.Module):
    def __init__(self):
        super(HugeLinearModel, self).__init__()
        self.linear1 = nn.Linear(1200, 1800)
        self.linear2 = nn.Linear(1800, 1500)
        self.linear3 = nn.Linear(1500, 2000)
        self.linear4 = nn.Linear(2000, 3000)

    def forward(self, x):
        output = self.linear1(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = HugeLinearModel()
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

# export to onnx
linear_path = "../huge_linear.onnx"
dummy_input = torch.randn(1, 1200)
onnx.export(model, dummy_input, linear_path, export_params=True, opset_version=11)

HugeLinearModel(
  (linear1): Linear(in_features=1200, out_features=1800, bias=True)
  (linear2): Linear(in_features=1800, out_features=1500, bias=True)
  (linear3): Linear(in_features=1500, out_features=2000, bias=True)
  (linear4): Linear(in_features=2000, out_features=3000, bias=True)
)
Model weights:
linear1.weight: torch.Size([1800, 1200])
  Weight values (first 5): tensor([ 0.0016,  0.0171, -0.0018, -0.0248,  0.0098], grad_fn=<SliceBackward0>)
linear1.bias: torch.Size([1800])
  Bias values (first 5): tensor([ 0.0124, -0.0057,  0.0237, -0.0210,  0.0045], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([1500, 1800])
  Weight values (first 5): tensor([-0.0028,  0.0201,  0.0133,  0.0030, -0.0084], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([1500])
  Bias values (first 5): tensor([-0.0092,  0.0094, -0.0049, -0.0170, -0.0226], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([2000, 1500])
  Weight values (first 5): tensor([ 0.0060,  0.0166,  0.0101, -0.0238, -0.0059], 